# Phase 5 — Sensitivity Analysis and Report

Runs documented scenarios and exports report-ready comparison tables.

In [1]:
from pathlib import Path
import sys
import geopandas as gpd
import pandas as pd
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.config import GEOJSON_DIR, TABLES_DIR
from src.analysis.phase5_delivery import default_scenarios, score_desert_scenario, rank_candidates, summarize_candidate_stability, severity_sensitivity, scenario_metadata
deserts = gpd.read_file(GEOJSON_DIR / 'charging_deserts.geojson')
candidates = pd.read_csv(TABLES_DIR / 'conversion_candidates.csv')
print('Desert tracts:', len(deserts), 'Candidates:', len(candidates))

Desert tracts: 9129 Candidates: 9591


In [2]:
scenarios = default_scenarios()
scenario_metadata(scenarios).to_csv(TABLES_DIR / 'sensitivity_scenarios.csv', index=False)
tract_results = {}
unsupported = {}
for scenario in scenarios:
    try:
        tract_results[scenario.name] = score_desert_scenario(deserts, scenario)
    except KeyError as exc:
        unsupported[scenario.name] = str(exc)
print('Skipped scenarios:', unsupported or 'none')
severity_summary = severity_sensitivity(tract_results)
severity_summary.to_csv(TABLES_DIR / 'severity_sensitivity.csv', index=False)
severity_summary

Skipped scenarios: none


,scenario,tract_count,adequate,moderate_desert,median_score,max_score
0,baseline,9129,8228,901,42.28,72.54
1,ev_gap_high,9129,7683,1446,44.22,73.46
2,dcfc_distance_high,9129,8391,738,41.69,71.47
3,gas_access_high,9129,8220,909,41.84,71.99
4,gas_buffer_5km,9129,8228,901,42.28,72.54
5,redundancy_2km,9129,8228,901,42.28,72.54
6,dcfc_only_supply,9129,8164,965,42.40,72.54
7,per_capita_supply,9129,8161,968,42.41,72.54


In [3]:
ranked = [rank_candidates(candidates, s.name) for s in scenarios]
stability = summarize_candidate_stability(ranked)
stability.to_csv(TABLES_DIR / 'candidate_stability.csv', index=False)
stability.head(20)

,station_id,score_baseline,rank_baseline,score_ev_gap_high,rank_ev_gap_high,score_dcfc_distance_high,rank_dcfc_distance_high,score_gas_access_high,rank_gas_access_high,score_gas_buffer_5km,...,score_dcfc_only_supply,rank_dcfc_only_supply,score_per_capita_supply,rank_per_capita_supply,top20_count,top50_count,average_rank,worst_rank,rank_std,stable_candidate
9501,9574,70.578646,1,70.578646,1,70.578646,1,70.578646,1,70.578646,...,70.578646,1,70.578646,1,8,8,1.0,1,0.0,True
6682,7024,68.718913,2,68.718913,2,68.718913,2,68.718913,2,68.718913,...,68.718913,2,68.718913,2,8,8,2.0,2,0.0,True
1723,2550,68.409388,3,68.409388,3,68.409388,3,68.409388,3,68.409388,...,68.409388,3,68.409388,3,8,8,3.0,3,0.0,True
7405,7677,68.188614,4,68.188614,4,68.188614,4,68.188614,4,68.188614,...,68.188614,4,68.188614,4,8,8,4.0,4,0.0,True
8061,8268,68.188614,5,68.188614,5,68.188614,5,68.188614,5,68.188614,...,68.188614,5,68.188614,5,8,8,5.0,5,0.0,True
7636,7885,67.365568,6,67.365568,6,67.365568,6,67.365568,6,67.365568,...,67.365568,6,67.365568,6,8,8,6.0,6,0.0,True
3311,3986,67.248646,7,67.248646,7,67.248646,7,67.248646,7,67.248646,...,67.248646,7,67.248646,7,8,8,7.0,7,0.0,True
9196,9297,67.248646,8,67.248646,8,67.248646,8,67.248646,8,67.248646,...,67.248646,8,67.248646,8,8,8,8.0,8,0.0,True
6413,6783,67.101263,9,67.101263,9,67.101263,9,67.101263,9,67.101263,...,67.101263,9,67.101263,9,8,8,9.0,9,0.0,True
8719,8862,66.809537,10,66.809537,10,66.809537,10,66.809537,10,66.809537,...,66.809537,10,66.809537,10,8,8,10.0,10,0.0,True


## Interpretation checklist

Confirm baseline consistency, report severity changes, identify stable candidates, and describe results as model-based prioritization rather than feasibility determinations.